# bridge_report

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): the analysis. `c1Summaries` reports every `C1_PLAN` item from the full grid (new and reused boards): Gate C1 (`gateC1`: part i, `bridge1` against one board of 64 with guards, a one-sided exact sign-flip over 8 boards; part ii, the margin of half the uncoupled squads' lead), its robustness on raw honest welfare, H20-3 to H20-5 and H20-7 on seeds 1800-1804 (bootstrap intervals, uncorrected sign-flip p), H20-8's cost without liars on the clean boards, the inert-bridge flag, and per arm the Phase 18, bridge and core measures; it refuses rows unless every planned cell is present. `partBSummaries`: part b (H20-6, 3B at 32 agents, squads of 4 against one board). The report, the figure and synthetic rows are in `bridge_write`.

In [ ]:
#| default_exp bridge_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Gate C1 and the summaries

In [ ]:
#| export
import numpy as np

from explRL.bridge_grid import ARMS20, PARTB_MODELS
from explRL.bridge_mech import coreMeasures
from explRL.bridge_plan import C1_PLAN
from explRL.colony4_report import CELL_KEYS
from explRL.colony_report import NAN, bootSeeds, cellVals, pairs, signFlipExact
from explRL.rl17_grid import BIG_SEEDS
from explRL.rl17_report import boards
from explRL.rl19_measures import index

IND, CLEAN, N20 = 'independent', 'clean', 64
BRIDGE_COLS = ('mirrorPosts', 'mirrorFalse', 'bridgeRelays', 'relaysPerBridgeTurn', 'falseRelayShare', 'relayLag', 'relaysUsed',
               'crossContamination', 'coupledSolveShare', 'honestPooledCoupled', 'honestPooledUncoupled', 'bridgeOpens', 'bridgeSolves',
               'bridgeGuardPostsPerTurn', 'inTokPerCallBridge', 'inTokPerCallHolder', 'inTokPerCallOther')


def _m(v):
    "Mean of the finite values (NaN, silently, without any)."
    v = [x for x in map(float, v) if np.isfinite(x)]
    return float(np.mean(v)) if v else NAN


def planned20(plan=C1_PLAN):
    "{(level, seed)} of every planned cell at N 64: the new cells with liars and clean (P20-14) and the reused references (P20-6)."
    c = plan['cells']
    out = {(f'{a}|{IND}', s) for a, ss in c['liars'].items() for s in ss} | {(f'{a}|{CLEAN}', s) for a, ss in c['clean'].items() for s in ss}
    return out | {(f'{a}|{IND}', s) for a, d in c['reused'].items() for s in d['seeds']}


def _d(rows, arm, base, metric='honestPooled', fault=IND, seeds=None):
    "{(seed, 64): arm - base}, paired on seed (colony_report.pairs), on `seeds` (default every seed both have)."
    return {k: v for k, v in pairs(rows, arm, metric, fault, base, sizes=[N20]).items() if seeds is None or k[0] in seeds}


def _test(dp, nboot, tail='greater'):
    "Estimate (mean), bootSeeds' 95% interval and the uncorrected exact one-sided sign-flip p (tail 'greater' or 'less')."
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    return dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(list(dp.values()), tail))


def _lin(a, b, w):
    "The per-board margin a - w x b on the boards both have."
    return {k: a[k] - w * b[k] for k in a if k in b}


def _ratio(a, b, nboot, seed=0):
    "mean(a) / mean(b) over the boards both have, and its 95% interval resampling boards (as bootSeeds)."
    ks = sorted(set(a) & set(b))
    if not ks:
        return NAN, [NAN, NAN]
    r = lambda kk: (lambda d: np.mean([a[k] for k in kk]) / d if d else NAN)(np.mean([b[k] for k in kk]))
    rng = np.random.default_rng(seed)
    bs = [r([ks[j] for j in rng.integers(0, len(ks), len(ks))]) for _ in range(nboot)]
    bs = [x for x in bs if np.isfinite(x)]
    return float(r(ks)), [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))] if bs else [NAN, NAN]


def gateC1(rows, plan=C1_PLAN, nboot=2000, metric='honestPooled'):
    """Gate C1 on `metric` over the 8 boards (seeds 1800-1807, P20-7). Part (i): bridge1 - colonyGuards, one-sided exact
    sign-flip, passes at p < alpha with a positive estimate. Part (ii), a margin, not a test: mean(bridge1 - colonyGuards) >=
    0.5 x mean(squads - colonyGuards) (the lead), with the per-board margin's interval and uncorrected sign-flip p."""
    s8 = set(plan['seeds']['eight'])
    b, sq = _d(rows, 'bridge1', 'colonyGuards', metric, seeds=s8), _d(rows, 'squads', 'colonyGuards', metric, seeds=s8)
    i = _test(b, nboot)
    i['passes'] = bool(i['p'] < plan['gate']['alpha'] and i['estimate'] > 0)
    lead = _test(sq, nboot)
    ii = dict(_test(_lin(b, sq, 0.5), nboot), lead=lead, bridge=i['estimate'], bar=0.5 * lead['estimate'])
    ii['holds'] = bool(i['estimate'] >= ii['bar'])
    return dict(metric=metric, i=i, ii=ii, passes=bool(i['passes'] and ii['holds']))


def _cell(rows, arm, fault, k):
    return _m(r[k] for r in rows if r['level'] == f'{arm}|{fault}' and k in r and r[k] not in ('', None))


def c1Summaries(rows, plan=C1_PLAN, nboot=None):
    """Every C1_PLAN item from the full grid's rows (new and reused; FR-D2): Gate C1 (parts i and ii) on honestPooled with its
    robustness on welfareHonest; H20-3 to H20-5 and H20-7 on seeds 1800-1804 (estimates, bootstrap intervals, uncorrected
    sign-flip p); H20-8, cost without liars on the clean boards (descriptive); the inert-bridge flag; and per arm its honest
    pooled share, advantage over colonyGuards, the Phase 18 and bridge columns and the core measures (FR-D5). ValueError
    unless every planned cell, clean cells included, is present; other rows (the pilot's) are left out."""
    want, nboot = planned20(plan), nboot or plan['nboot']
    key = lambda r: (r['level'], int(float(r['seed'])))
    rows = [r for r in rows if int(float(r['N'])) == N20 and key(r) in want]
    if miss := sorted(want - {key(r) for r in rows}):
        raise ValueError(f'{len(miss)} planned C1 cells missing, e.g. {miss[:4]}')
    gate, rob = gateC1(rows, plan, nboot), gateC1(rows, plan, nboot, 'welfareHonest')
    gate['robust'] = {p: bool(np.sign(rob[p]['estimate']) == np.sign(gate[p]['estimate'])) for p in ('i', 'ii')}
    gate['welfareHonest'] = rob
    s5 = set(plan['seeds']['five'])
    d = lambda a, b, m='honestPooled', f=IND: _d(rows, a, b, m, f, s5)
    b1c, sqc, b21 = d('bridge1', 'coupled'), d('squads', 'coupled'), d('bridge2', 'bridge1')
    wb, wbCi = _ratio(b1c, sqc, nboot)
    H3 = dict(_test(_lin(b1c, sqc, 0.5), nboot), winBack=wb, winBackCi=wbCi, couplingCost=_test(sqc, nboot), bridgeGain=_test(b1c, nboot))
    H3['supported'] = bool(H3['couplingCost']['estimate'] > 0 and wb > 0.5)
    H4 = dict(_test(_lin(b21, sqc, 0.25), nboot, 'less'), secondBridge=_test(b21, nboot))
    H4['supported'] = bool(H4['estimate'] < 0)
    H5 = dict(_test(d('bridge1', 'bridgeGuard', 'falseRelayShare'), nboot), pooled=_test(d('bridgeGuard', 'bridge1'), nboot))
    H5['supported'] = bool(H5['estimate'] > 0)
    coupledArms = [a for a in ARMS20 if ARMS20[a].get('couple')]
    H7 = dict(perArm={a: bootSeeds({k: v for k, v in cellVals(rows, a, IND, 'crossContamination', [N20]).items() if np.isfinite(v)},
                                   nboot=nboot) for a in coupledArms},
              guarding=_test(d('bridge1', 'bridgeGuard', 'crossContamination'), nboot))
    clean = list(plan['cells']['clean'])
    H8 = dict(perArm={a: {m: _cell(rows, a, CLEAN, m) for m in ('honestPooled', 'welfareHonest')} for a in clean},
              contrasts={f'{a} - squads': {m: _test(d(a, 'squads', m, CLEAN), nboot, 'less') for m in ('honestPooled', 'welfareHonest')}
                         for a in clean if a != 'squads'})
    hyp = dict(H20_3=H3, H20_4=H4, H20_5=H5, H20_7=H7, H20_8=H8)
    inert = dict(estimate=H3['bridgeGain']['estimate'], fires=bool(gate['passes'] and not H3['bridgeGain']['estimate'] > 0))
    ix, arms = index(rows), {}
    for a in ARMS20:
        hp = {k: v for k, v in cellVals(rows, a, IND, 'honestPooled', [N20]).items() if np.isfinite(v)}
        arms[a] = dict(honestPooled=bootSeeds(hp, nboot=nboot), boards=len(hp),
                       advantage=_test(_d(rows, a, 'colonyGuards', seeds=set(plan['seeds']['eight'])), nboot) if a != 'colonyGuards' else None,
                       columns={k: _cell(rows, a, IND, k) for k in CELL_KEYS + BRIDGE_COLS}, core=coreMeasures(ix, a, IND),
                       coreClean=coreMeasures(ix, a, CLEAN) if a in clean else None)
    return dict(gate=gate, hyp=hyp, inert=inert, arms=arms, boards=len(rows))

## Part b (H20-6)

In [ ]:
#| export
P17_GAIN = {4: 0.078, 32: 0.006}                               # Phase 17's H17-1 gains at 4 and 32 agents (shown beside H20-6)
PARTB_TESTS = dict(squads4=('n32sq4', 'n32sq4liar'), oneBoard=('n32', 'n32liar'))


def partBSummaries(rows, models=PARTB_MODELS, warm='warm', seeds=BIG_SEEDS):
    """Part b (FR-D3, H20-6, descriptive) at 32 agents: per run, the welfare gain over the warm start on clean boards (mean over
    the seeds of run - warm) in squads of 4 and on one board, their difference, and the honest loss with liars (welfareHonest
    clean - with 8 liars, per seed) beside each; the difference's sign-flip over the runs (one-sided); Phase 17's gains beside.
    ValueError unless every model has every level on every seed."""
    miss = [f'{m}|{t}' for m in models for ts in PARTB_TESTS.values() for t in ts if set(boards(rows, m, t, 'welfare')) < set(seeds)]
    if miss:
        raise ValueError(f'part b rows missing: {miss[:4]}')
    gain = lambda m, t: _m(v - w[s] for w in [boards(rows, warm, t, 'welfare')] for s, v in boards(rows, m, t, 'welfare').items() if s in w)
    loss = lambda m, c, l: _m(v - L[s] for L in [boards(rows, m, l, 'welfareHonest')] for s, v in boards(rows, m, c, 'welfareHonest').items() if s in L)
    runs = {m: {a: dict(welfare=_m(boards(rows, m, c, 'welfare').values()), gain=gain(m, c), honestLoss=loss(m, c, l))
                for a, (c, l) in PARTB_TESTS.items()} for m in models}
    for v in runs.values():
        v['diff'] = v['squads4']['gain'] - v['oneBoard']['gain']
    ad = [m for m in models if m != warm]
    col = lambda f: [f(runs[m]) for m in ad]
    diff = col(lambda v: v['diff'])
    return dict(runs=runs, H20_6=dict(n=len(ad), estimate=_m(diff), p=signFlipExact(diff), positive=sum(x > 0 for x in diff),
                                      **{f'gain_{a}': _m(col(lambda v, a=a: v[a]['gain'])) for a in PARTB_TESTS},
                                      **{f'honestLoss_{a}': _m(col(lambda v, a=a: v[a]['honestLoss'])) for a in PARTB_TESTS}),
                phase17=P17_GAIN, descriptive=True)

## Tests

In [ ]:
# test_c1_report (AC-9, FR-D2, FR-D3, FR-D4): on syntheticRows20 with known effects, the gate's parts (i) and (ii) with their
# robustness, the margin, win-back (H20-3), H20-4, H20-5, H20-7, the clean contrasts of H20-8, the inert-bridge flag and the
# core measures (FR-D5) are recovered exactly (one noise draw per seed is shared by every arm, so paired differences are
# exact: p 1/256 over 8 boards, 1/32 over 5); the gate fails on either part, the flag fires when bridges do nothing and the
# gate passes; missing cells are refused; partBSummaries recovers H20-6 on synthetic part-b rows; the report names every
# C1_PLAN item and c1.png is written; rows read back from CSV give the same summary
import tempfile
from pathlib import Path

from explRL.bench_grid import readCSV, writeCSV
from explRL.bridge_write import figC1, syntheticPartB, syntheticRows20, writeC1

close = lambda a, b: abs(a - b) < 1e-9
rows = syntheticRows20()
assert len(rows) == 54 and len(planned20()) == 54                     # 23 new with liars, 15 clean, 16 reused
S = c1Summaries(rows + [dict(rows[0], seed=1890, level='bridge1|independent')], nboot=200)   # a pilot row is left out
G, H = S['gate'], S['hyp']
assert S['boards'] == 54 and G['metric'] == 'honestPooled' and G['passes']
# part (i): bridge1 - colonyGuards = lead - cost + back = 0.141 on all 8 boards, so p = 1/256
assert close(G['i']['estimate'], 0.171 - 0.10 + 0.07) and G['i']['n'] == 8 and G['i']['p'] == 1 / 256 and G['i']['passes']
# part (ii): the lead 0.171, the bar half of it, the per-board margin 0.141 - 0.0855
assert close(G['ii']['lead']['estimate'], 0.171) and close(G['ii']['bar'], 0.0855) and close(G['ii']['estimate'], 0.141 - 0.0855)
assert G['ii']['holds'] and G['ii']['p'] == 1 / 256 and G['ii']['ci'][0] <= G['ii']['estimate'] + 1e-12 <= G['ii']['ci'][1] + 2e-12
# robustness on welfareHonest (1.3 x honestPooled in the synthetic rows): same signs
assert G['robust'] == dict(i=True, ii=True) and close(G['welfareHonest']['i']['estimate'], 1.3 * 0.141)
# H20-3: coupling cost 0.10, bridge gain 0.07, win-back 0.7 (> 0.5), the per-board margin 0.07 - 0.05 with p 1/32
h3 = H['H20_3']
assert close(h3['couplingCost']['estimate'], 0.10) and close(h3['bridgeGain']['estimate'], 0.07) and close(h3['winBack'], 0.7)
assert close(h3['estimate'], 0.02) and h3['p'] == 1 / 32 and h3['n'] == 5 and h3['supported']
assert h3['winBackCi'][0] <= 0.7 + 1e-9 and h3['winBackCi'][1] >= 0.7 - 1e-9
# H20-4: bridge2 - bridge1 = 0.01 < 0.25 x 0.10, margin -0.015 (p for < 0: 1/32)
assert close(H['H20_4']['secondBridge']['estimate'], 0.01) and close(H['H20_4']['estimate'], -0.015)
assert H['H20_4']['p'] == 1 / 32 and H['H20_4']['supported']
# H20-5: false-relay share 0.3 against 0.1 with guarding bridges; honestPooled bridgeGuard - bridge1 = 0.005
assert close(H['H20_5']['estimate'], 0.2) and H['H20_5']['p'] == 1 / 32 and H['H20_5']['supported']
assert close(H['H20_5']['pooled']['estimate'], 0.005)
# H20-7: contamination per coupled arm, and bridge1 - bridgeGuard = 0.05
assert {a: round(v[0], 9) for a, v in H['H20_7']['perArm'].items()} == dict(coupled=0.0, bridge1=0.08, bridge2=0.08, bridgeGuard=0.03)
assert close(H['H20_7']['guarding']['estimate'], 0.05) and H['H20_7']['guarding']['n'] == 5
# H20-8: clean boards, coupled - squads = -0.03 and bridge1 - squads = -0.01 (p for < 0: 1/32), welfareHonest 1.3 times
c8 = H['H20_8']['contrasts']
assert set(c8) == {'coupled - squads', 'bridge1 - squads'} and set(H['H20_8']['perArm']) == {'squads', 'coupled', 'bridge1'}
assert close(c8['coupled - squads']['honestPooled']['estimate'], -0.03) and close(c8['bridge1 - squads']['honestPooled']['estimate'], -0.01)
assert close(c8['coupled - squads']['welfareHonest']['estimate'], -0.039)
assert all(t['p'] == 1 / 32 and t['n'] == 5 for d in c8.values() for t in d.values())
assert S['inert'] == dict(estimate=h3['bridgeGain']['estimate'], fires=False)
# per arm: boards, advantage over colonyGuards, core measures equal their definitions on the synthetic columns
A = S['arms']
assert {a: v['boards'] for a, v in A.items()} == dict(squads=8, colonyGuards=8, coupled=5, bridge1=8, bridge2=5, bridgeGuard=5)
assert A['colonyGuards']['advantage'] is None and close(A['coupled']['advantage']['estimate'], 0.071)
core = A['bridge1']['core']
assert core['boards'] == 8 and close(core['gullibility'], 6 / 1920) and close(core['suppression'], 0.5) and close(core['contestsPerLie'], 0.5)
assert core['timeToContest'] == 2.0 and close(core['guardedLies'], 0.4) and close(core['errorCorr'], 1 / 3) and core['tokPerSolved'] == 60000.0
assert 0 < core['dol'] < 1 and close(core['crossContamination'], 0.08) and A['bridge1']['coreClean']['boards'] == 5 and A['bridge2']['coreClean'] is None
assert close(A['bridge1']['columns']['falseRelayShare'], 0.3) and A['squads']['columns']['bridgeRelays'] == 0.0
# the inert-bridge flag: a small coupling cost and bridges that do nothing; the gate passes, the flag fires
Si = c1Summaries(syntheticRows20(cost=0.03, back=-0.005), nboot=50)
assert Si['gate']['passes'] and Si['inert']['fires'] and close(Si['inert']['estimate'], -0.005) and not Si['hyp']['H20_3']['supported']
# part (ii) fails (bridge1 - colonyGuards 0.021 < 0.0855) while part (i) passes; then part (i) fails too
Sf = c1Summaries(syntheticRows20(cost=0.15, back=0.0), nboot=50)
assert Sf['gate']['i']['passes'] and not Sf['gate']['ii']['holds'] and not Sf['gate']['passes'] and not Sf['inert']['fires']
Sf = c1Summaries(syntheticRows20(cost=0.25, back=0.0), nboot=50)
assert not Sf['gate']['i']['passes'] and Sf['gate']['i']['p'] == 1.0 and not Sf['gate']['passes']
# every planned cell, clean cells included, must be present
for drop in ('bridge1|clean', 'colonyGuards|independent'):
    try:
        c1Summaries([r for r in rows if not (r['level'] == drop and r['seed'] == 1803)], nboot=10)
        raise AssertionError(drop)
    except ValueError as e:
        assert 'missing' in str(e) and drop in str(e)
# part b: each run's difference is exactly gain4 - gain1 = 0.044 (p 1/256); honest losses 0.04 and 0.08
B = partBSummaries(syntheticPartB())
h6 = B['H20_6']
assert close(h6['estimate'], 0.044) and h6['p'] == 1 / 256 and h6['n'] == 8 and h6['positive'] == 8
assert close(h6['honestLoss_squads4'], 0.04) and close(h6['honestLoss_oneBoard'], 0.08) and B['phase17'] == {4: 0.078, 32: 0.006}
assert close(h6['gain_squads4'] - h6['gain_oneBoard'], 0.044) and close(B['runs']['warm']['squads4']['gain'], 0.0)
assert partBSummaries(syntheticPartB(gain4=0.0, gain1=0.03))['H20_6']['p'] == 1.0
try:
    partBSummaries([r for r in syntheticPartB() if r['level'] != 'r17mix_s3|n32liar'])
    raise AssertionError('missing part b rows')
except ValueError as e:
    assert 'r17mix_s3|n32liar' in str(e)
# the report, the figure, and a CSV round trip
with tempfile.TemporaryDirectory() as d:
    writeC1(Path(d) / 'report.md', S, dict(planHash='x', commit='y', mode='quick'), partB=B)
    figC1(d, S)
    txt = (Path(d) / 'report.md').read_text()
    assert (Path(d) / 'c1.png').stat().st_size > 10_000 and '**Gate C1: passes**' in txt and 'does not fire' in txt
    assert all(f'- {k}:' in txt for k in C1_PLAN) and all(f'H20-{i}' in txt for i in (3, 4, 5, 6, 7, 8)) and 'gullibility' in txt
    writeC1(Path(d) / 'b.md', None, dict(mode='partb'), partB=B)
    assert 'Part b' in (Path(d) / 'b.md').read_text() and '## Gate C1' not in (Path(d) / 'b.md').read_text()
    writeCSV(rows, Path(d) / 'runs.csv')
    S2 = c1Summaries(readCSV(Path(d) / 'runs.csv'), nboot=200)
assert close(S2['gate']['ii']['estimate'], G['ii']['estimate']) and close(S2['arms']['bridge1']['core']['dol'], core['dol'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()